# Closed-Loop (Thin Colab Notebook)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/thesis-research-colab/blob/main/notebooks/closedloop_simulation_colab.ipynb)

This notebook is orchestration-first: heavy logic is in `src/closedloop/*.py`, while notebook cells stay focused on run controls and interpretation.


## Notebook Purpose And Contract

This notebook is designed for fast patch-and-test loops in Colab.

Execution contract:
1. Sync latest repo code into `/content/thesis-research-colab`.
2. Validate Drive access for persistent artifacts.
3. Run deterministic environment setup only when required.
4. Run diagnostics first (quick probe, preflight, calibration, surprise gate).
5. Execute full optimization only when diagnostics pass policy.

After any new Git push:
1. Re-run Step 1 (repo sync).
2. Re-run Step 4 (imports/hot-reload).
3. Re-run from Step 5 onward.


## Step 1 - Sync Repository

Goal: make the runtime match latest `main` before any experiment logic.

This step:
- clones the repo if missing,
- fast-forwards existing checkout to current `main`,
- restores `sys.path` for `src` imports.

Expected output:
- `Repo commit: <sha>`
- `Has src/closedloop: True`

If this step fails, fix git auth/network first and do not continue.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/achyutmorang/thesis-research-colab.git"
REPO_DIR = "/content/thesis-research-colab"
REPO_BRANCH = "main"

content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))

if not Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

for path in (REPO_DIR, f"{REPO_DIR}/src"):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import ensure_repo_checkout

repo_sync = ensure_repo_checkout(repo_url=REPO_URL, repo_dir=REPO_DIR, branch=REPO_BRANCH)
globals()['_CLOSEDLOOP_REPO_REV'] = str(repo_sync.repo_rev)

print("Working directory:", os.getcwd())
print("Repo commit:", repo_sync.repo_rev)
print("Has src/closedloop:", os.path.exists(os.path.join(REPO_DIR, "src", "closedloop")))
print("sys.path[0:3]:", list(repo_sync.sys_path_head[:3]))


## Step 2 - Mount Persistent Storage

Goal: ensure resume/checkpoint/export artifacts survive Colab runtime loss.

This step verifies:
- Drive mount health,
- required folder existence (`/content/drive/MyDrive/waymax_experiments`),
- write access (temporary probe file).

If this fails with permission errors, request edit access to the shared folder and rerun this step.


In [ ]:
from pathlib import Path

from src.platform import ensure_drive_ready

REQUIRED_DRIVE_FOLDER = Path('/content/drive/MyDrive/waymax_experiments')
VERIFY_DRIVE_ACCESS_EVERY_RUN = False

status = ensure_drive_ready(
    required_drive_folder=str(REQUIRED_DRIVE_FOLDER),
    verify_drive_access_every_run=bool(VERIFY_DRIVE_ACCESS_EVERY_RUN),
)

if not status.is_colab:
    print('[drive] non-Colab environment; skipping mount')


## Step 3 - Deterministic Runtime Setup

Goal: converge package/runtime state to required pinned versions.

Setup behavior:
- uses cache when lockfile + runtime fingerprint are unchanged,
- reinstalls only when needed,
- auto-restarts runtime only if compiled packages changed.

Important:
- if auto-restart happens, rerun from Step 1.
- no manual `RUN_SETUP` toggle is required.


In [ ]:
from pathlib import Path

from src.platform import run_cached_deterministic_setup

FORCE_REINSTALL = False
AUTO_RESTART_AFTER_SETUP = True
STRICT_LOCKFILE_CHECK = True
SETUP_CACHE_ENABLED = True
REVALIDATE_CORE_IMPORTS_ON_CACHE_HIT = True
SETUP_CACHE_PATH = Path('/content/.closedloop_setup_cache.json')

setup_bundle = run_cached_deterministic_setup(
    repo_root='.',
    force_reinstall=bool(FORCE_REINSTALL),
    auto_restart_after_setup=bool(AUTO_RESTART_AFTER_SETUP),
    strict_lockfile_check=bool(STRICT_LOCKFILE_CHECK),
    setup_cache_enabled=bool(SETUP_CACHE_ENABLED),
    revalidate_core_imports_on_cache_hit=bool(REVALIDATE_CORE_IMPORTS_ON_CACHE_HIT),
    setup_cache_path=str(SETUP_CACHE_PATH),
    repo_rev=str(globals().get('_CLOSEDLOOP_REPO_REV', 'unknown')),
)
setup_result = setup_bundle.result
print('[setup] result:', setup_result)

if bool(setup_result.get('restart_required', False)) and (not AUTO_RESTART_AFTER_SETUP):
    print('[setup] restart_required=True; please restart runtime before running simulation cells.')


### Post-Restart Verification (Recommended)

After auto-restart, quickly verify interpreter + numeric stack before long runs.

```python
import sys, numpy as np
print("kernel:", sys.executable)
print("numpy:", np.__version__, np.__file__)
from numpy._core.umath import _center, _expandtabs
print("NumPy private-symbol probe: OK")
```


### Optional Manual Checkpoint Fallback

Use this only if automatic checkpoint retrieval fails.


In [ ]:
# Manual checkpoint fallback (run only if setup could not fetch checkpoint)
# !mkdir -p /content/checkpoints
# !wget -O /content/checkpoints/lantentdriver_t2_J3.ckpt https://huggingface.co/Sephirex-x/LatentDriver/resolve/main/checkpoints/lantentdriver_t2_J3.ckpt
# !ls -lh /content/checkpoints/lantentdriver_t2_J3.ckpt


## Step 4 - Import Modular Pipeline APIs

Goal: import modular helpers from shared packages:
- `src.platform` for Colab/runtime lifecycle,
- `src.workflows` for reusable notebook orchestration,
- `src.closedloop` for domain-specific simulation logic.

This step hot-reloads `src.*` modules after each git pull so notebook state tracks latest pushed code.


In [ ]:
import os
import sys

from src.platform import prepare_repo_imports

REPO_DIR = '/content/thesis-research-colab'
FORCE_MODULE_HOT_RELOAD = True

prepare_repo_imports(repo_dir=REPO_DIR, force_module_hot_reload=bool(FORCE_MODULE_HOT_RELOAD))

try:
    from src.workflows import (
        analyze_signal_if_available,
        initialize_run_context,
        report_export_bundle,
        report_main_loop_bundle,
        report_preflight_bundle,
        report_quick_probe_bundle,
        report_run_context,
        report_signal_bundle,
        report_surprise_gate_bundle,
        run_main_loop_with_policy,
        run_preflight_bundle,
        run_quick_probe_with_auto_escalation,
        run_surprise_gate_with_policy,
        summarize_and_export_if_available,
    )
except ModuleNotFoundError as e:
    msg = (
        'Import failed for src.workflows/src.platform. Re-run Step 1, then rerun this cell.\n'
        f'cwd={os.getcwd()}\n'
        f'sys.path_head={sys.path[:5]}'
    )
    raise RuntimeError(msg) from e


## Step 5 - Configure Run Identity, Persistence, And Auto-Run Policy

This step sets experiment identity and run controls:
- `RUN_TAG`: unique experiment name,
- `PERSIST_ROOT`: persistent Drive root,
- `N_SHARDS`, `SHARD_ID`: workload partition/resume routing,
- auto-run policy for full closed-loop stage.

`SHARD_ID="auto"` resumes the least-complete shard first.


In [ ]:
RUN_MAIN_LOOP_OVERRIDE = None
AUTO_RUN_MAIN_LOOP_WHEN_READY = True

RUN_TAG = 'closedloop_v1'
PERSIST_ROOT = '/content/drive/MyDrive/waymax_experiments/closedloop_runs/v1'
N_SHARDS = 5
SHARD_ID = 'auto'
RESTORE_FROM_UPLOAD = False
SHARD_ID_REQUESTED = SHARD_ID

run_context = initialize_run_context(
    run_tag=RUN_TAG,
    persist_root=PERSIST_ROOT,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    auto_run_main_loop_when_ready=bool(AUTO_RUN_MAIN_LOOP_WHEN_READY),
    run_main_loop_override=RUN_MAIN_LOOP_OVERRIDE,
)

cfg = run_context.cfg
search_cfg = run_context.search_cfg
ckpt_scan_df = run_context.ckpt_scan_df
shard_progress_df = run_context.shard_progress_df
SHARD_ID = run_context.shard_id
run_prefix = run_context.run_prefix

if isinstance(SHARD_ID_REQUESTED, str) and SHARD_ID_REQUESTED.strip().lower() == 'auto':
    print(f'[shard] auto-selected SHARD_ID={SHARD_ID}')
else:
    print(f'[shard] using SHARD_ID={SHARD_ID}')

report_run_context(run_context, display_fn=display)


## Step 6 - Build Dataset, Splits, And Scenario Handles

This step runs an optional quick surprise probe, then builds full run context:
- reference/eval splits,
- scenario handles,
- base open-loop baseline frame.

Probe escalation attempts automatically increase perturbation intensity when collapse is detected.


In [ ]:
RUN_QUICK_SURPRISE_PROBE = True
quick_probe_settings = {
    'quick_probe_scenarios': 1,
    'quick_probe_proposals_per_scenario': 4,
    'stop_if_quick_probe_collapsed': False,
    'auto_escalate_quick_probe': True,
    'max_probe_escalations': 3,
    'probe_scale_multipliers': (1.0, 1.35, 1.8),
    'probe_delta_l2_multipliers': (1.0, 1.2, 1.4),
    'probe_delta_clip_multipliers': (1.0, 1.1, 1.2),
    'probe_budget_bump_per_escalation': 2,
    'apply_successful_probe_tuning': True,
}

probe_bundle = run_quick_probe_with_auto_escalation(
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    run_quick_surprise_probe_enabled=bool(RUN_QUICK_SURPRISE_PROBE),
    **quick_probe_settings,
)

cfg = probe_bundle.cfg
search_cfg = probe_bundle.search_cfg
runner = probe_bundle.runner
eval_idx = probe_bundle.eval_idx
reference_df = probe_bundle.reference_df
base_eval_openloop_df = probe_bundle.base_eval_openloop_df
reference_idx = probe_bundle.reference_idx
candidate_eval_idx = probe_bundle.candidate_eval_idx

report_quick_probe_bundle(probe_bundle, search_cfg=search_cfg, display_fn=display)


In [ ]:
print(f"reference={len(reference_idx)} eval_pool={len(candidate_eval_idx)} eval_shard={len(eval_idx)}")


## Step 7 - Preflight Checks And Closed-Loop Calibration

Preflight validates planner and rollout health before expensive optimization.
Calibration computes thresholds/scales used by objective shaping and gating.

If preflight fails, optimization is skipped until checks pass.


In [ ]:
STOP_ON_PREFLIGHT_FAIL = False

preflight_bundle = run_preflight_bundle(
    runner=runner,
    cfg=cfg,
    search_cfg=search_cfg,
    eval_idx=eval_idx,
    reference_df=reference_df,
    restore_from_upload=RESTORE_FROM_UPLOAD,
    stop_on_preflight_fail=bool(STOP_ON_PREFLIGHT_FAIL),
)

READY_FOR_MAIN_LOOP = preflight_bundle.ready_for_main_loop
closedloop_calib_df = preflight_bundle.closedloop_calib_df
closedloop_thresholds = preflight_bundle.closedloop_thresholds
calib_diag_df = preflight_bundle.calib_diag_df
calib_quant_df = preflight_bundle.calib_quant_df
preflight_df = preflight_bundle.preflight_df

report_preflight_bundle(preflight_bundle, display_fn=display)


## Step 8 - Surprise Quality Gate

Gate purpose: block optimization when surprise signal appears degenerate.

Common failure patterns:
- near-zero surprise variance,
- zero nonzero-surprise fraction,
- excessive fallback/proxy behavior.


In [ ]:
SURPRISE_GATE_ENABLED = True
STOP_ON_GATE_FAIL = False
ALLOW_MAIN_LOOP_WHEN_GATE_FAILS = False

gate_bundle = run_surprise_gate_with_policy(
    closedloop_calib_df=closedloop_calib_df,
    ready_for_main_loop=bool(READY_FOR_MAIN_LOOP),
    surprise_gate_enabled=bool(SURPRISE_GATE_ENABLED),
    stop_on_gate_fail=bool(STOP_ON_GATE_FAIL),
    allow_main_loop_when_gate_fails=bool(ALLOW_MAIN_LOOP_WHEN_GATE_FAILS),
)

READY_FOR_OPTIMIZATION = gate_bundle.ready_for_optimization
report_surprise_gate_bundle(gate_bundle, display_fn=display)


## Step 9 - Closed-Loop Optimization Run (Auto-Gated)

Main loop runs only if gate policy allows it.

Override only when intentionally forcing behavior for debugging:
- `RUN_MAIN_LOOP_OVERRIDE=True` forces run,
- `RUN_MAIN_LOOP_OVERRIDE=False` forces skip,
- `None` uses diagnostic policy.


In [ ]:
main_loop_bundle = run_main_loop_with_policy(
    runner=runner,
    eval_idx=eval_idx,
    cfg=cfg,
    search_cfg=search_cfg,
    closedloop_thresholds=closedloop_thresholds,
    base_eval_openloop_df=base_eval_openloop_df,
    reference_df=reference_df,
    closedloop_calib_df=closedloop_calib_df,
    preflight_df=preflight_df,
    calib_diag_df=calib_diag_df,
    calib_quant_df=calib_quant_df,
    ready_for_optimization=bool(READY_FOR_OPTIMIZATION),
    auto_run_main_loop_when_ready=bool(AUTO_RUN_MAIN_LOOP_WHEN_READY),
    run_main_loop_override=RUN_MAIN_LOOP_OVERRIDE,
)

closedloop_results_df = main_loop_bundle.closedloop_results_df
closedloop_trace_df = main_loop_bundle.closedloop_trace_df
report_main_loop_bundle(main_loop_bundle, ready_for_optimization=bool(READY_FOR_OPTIMIZATION))


## Step 10 - Summarize And Export Artifacts

Exports include per-scenario outputs, traces, thresholds, diagnostics, and run metadata.
These are written under `PERSIST_ROOT` and used by evaluation notebooks and paper tables.


In [ ]:
export_bundle = summarize_and_export_if_available(
    cfg=cfg,
    search_cfg=search_cfg,
    eval_idx=eval_idx,
    closedloop_results_df=closedloop_results_df,
    closedloop_trace_df=closedloop_trace_df,
    base_eval_openloop_df=base_eval_openloop_df,
    reference_df=reference_df,
    closedloop_calib_df=closedloop_calib_df,
    preflight_df=preflight_df,
    calib_diag_df=calib_diag_df,
    calib_quant_df=calib_quant_df,
    closedloop_thresholds=closedloop_thresholds,
)

quick_summary_df = export_bundle.quick_summary_df
sanity_df = export_bundle.sanity_df
fairness_checks_df = export_bundle.fairness_checks_df
trace_diag_df = export_bundle.trace_diag_df
artifact_paths = export_bundle.artifact_paths

report_export_bundle(export_bundle, closedloop_results_df=closedloop_results_df, display_fn=display)


## Step 11 - Surprise Signal Usefulness Diagnostics

This step evaluates whether `surprise_pd` is useful for search outcomes (`delta_risk`):
- global correlation,
- quantile-bin trend behavior,
- top-k lift,
- within-scenario ranking consistency.


In [ ]:
signal_bundle = analyze_signal_if_available(
    closedloop_results_df=closedloop_results_df,
    n_bins=10,
    top_fracs=(0.10, 0.20),
    scenario_min_points=3,
)

signal_summary_df = signal_bundle.signal_summary_df
signal_method_corr_df = signal_bundle.signal_method_corr_df
signal_bin_df = signal_bundle.signal_bin_df
signal_topk_lift_df = signal_bundle.signal_topk_lift_df
signal_within_scenario_df = signal_bundle.signal_within_scenario_df

report_signal_bundle(signal_bundle, closedloop_results_df=closedloop_results_df, display_fn=display, preview_rows=20)
